<a href="https://colab.research.google.com/github/asharmaaryamani/product-reco-app/blob/main/Ikarus_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Colab cell 1: Setup
!pip -q install pandas numpy scikit-learn sentence-transformers transformers pinecone datasets pillow langchain langchain-community

import os, io, json, re, ast, requests, math, random
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from transformers import CLIPProcessor, CLIPModel
from PIL import Image

# Pinecone (v3)
!pip -q install pinecone
from pinecone import Pinecone, ServerlessSpec

# LangChain for GenAI
from langchain.prompts import PromptTemplate
from langchain.llms.huggingface_pipeline import HuggingFacePipeline
from transformers import pipeline

In [ ]:
# Colab cell 2: Load dataset (upload or from Drive)
# Option A: manual upload
from google.colab import files
uploaded = files.upload()  # upload intern_data_ikarus.csv

csv_path = '/content/intern_data_ikarus.csv'
df = pd.read_csv(csv_path)

# Basic cleanups
df['description'] = df['description'].fillna('')
df['brand'] = df['brand'].fillna('')
df['title'] = df['title'].fillna('')
df['price'] = df['price'].astype(str).str.replace(r'[^0-9\.]', '', regex=True).replace('', np.nan).astype(float)

# Parse categories and images columns if they are stringified lists
def parse_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return x if isinstance(x, list) else []

df['categories'] = df['categories'].apply(parse_list)
df['images'] = df['images'].apply(parse_list)

# Create a text field for embedding
def product_text(row):
    cats = ' > '.join(row['categories'][:4]) if isinstance(row['categories'], list) else ''
    return f"Title: {row['title']}\nBrand: {row['brand']}\nDescription: {row['description']}\nCategories: {cats}\nMaterial: {row.get('material','')}\nColor: {row.get('color','')}"
df['text_for_embed'] = df.apply(product_text, axis=1)

df.head(2)


In [ ]:
# Colab cell 3: Text embeddings
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embed_model.encode(df['text_for_embed'].tolist(), batch_size=64, show_progress_bar=True)
embeddings = np.array(embeddings, dtype=np.float32)

# KMeans clustering to group similar items
num_clusters = max(5, min(50, len(df)//200))  # heuristic
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init='auto')
df['cluster'] = kmeans.fit_predict(embeddings)

# Save clustering labels
df[['uniq_id','cluster']].to_csv('clusters.csv', index=False)


In [ ]:
# Colab cell 4: Pinecone init and upsert
from google.colab import userdata
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")  # set in Colab > Secrets
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "ikarus-products"
if index_name not in [idx['name'] for idx in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=embeddings.shape[1],
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(index_name)

# Prepare vectors
def meta_row(i):
    r = df.iloc[i]
    # Ensure metadata values are Pinecone-compatible
    metadata = {
        "uniq_id": str(r['uniq_id']),
        "title": r['title'],
        "brand": r['brand'],
        "description": r['description'],
        "categories": r['categories'] if isinstance(r['categories'], list) else [],
        "images": [str(img) for img in r['images']] if isinstance(r['images'], list) else [], # Ensure images are list of strings
        "material": str(r['material']) if pd.notna(r['material']) else '',
        "color": str(r['color']) if pd.notna(r['color']) else '',
        "cluster": int(r['cluster'])
    }
    # Only include price if it's not NaN
    if pd.notna(r['price']):
        metadata['price'] = float(r['price'])

    return metadata

batch = []
for i in range(len(df)):
    batch.append({"id": str(df.iloc[i]['uniq_id']), "values": embeddings[i].tolist(), "metadata": meta_row(i)})
    if len(batch) == 100:
        index.upsert(vectors=batch)
        batch = []
if batch:
    index.upsert(vectors=batch)

In [ ]:
import os
from google.colab import userdata

api_key = userdata.get('PINECONE_API_KEY')
if api_key is None:
    print("PINECONE_API_KEY secret is not set or not enabled for notebook access.")
else:
    print("PINECONE_API_KEY secret is set.")
    # You can optionally print a part of the key to confirm it's the correct one, but be cautious with exposing sensitive information.
    # print(f"First few characters of the key: {api_key[:5]}...")

In [ ]:
# Colab cell 5: Extract top-level category label
import torch
import joblib # Import joblib here as it's used in this cell

def top_category(cats):
    if isinstance(cats, list) and len(cats) > 0:
        return cats[0]
    return "Unknown"

df['label'] = df['categories'].apply(top_category)

# Download a subset of images for feasibility
subset = df[df['images'].apply(lambda x: isinstance(x, list) and len(x)>0)].copy()
subset = subset.sample(min(1200, len(subset)), random_state=42)

# CLIP feature extraction
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def fetch_image(urls):
    for u in urls:
        u = u.strip()
        try:
            img = Image.open(io.BytesIO(requests.get(u, timeout=10).content)).convert("RGB")
            return img
        except Exception:
            continue
    return None

X_feats, y_labels = [], []
for _, r in subset.iterrows():
    img = fetch_image(r['images'])
    if img is None:
        continue
    inputs = clip_proc(images=img, return_tensors="pt")
    with torch.no_grad():
        image_features = clip_model.get_image_features(**inputs)
    X_feats.append(image_features[0].cpu().numpy())
    y_labels.append(r['label'])

X = np.vstack(X_feats)
y = np.array(y_labels)

# Train/test split and linear classifier
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=2000)
clf.fit(Xtr, ytr)
print(classification_report(yte, clf.predict(Xte)))

# Save model artifacts
label_space = list(set(y)) # Assign the list of unique labels to label_space
joblib.dump(clf, 'clip_linear_classifier.joblib')
joblib.dump(label_space, 'label_space.joblib') # Dump the variable label_space
# No need to save CLIP weights (loaded on backend); alternatively cache HF models in backend

In [ ]:
# Colab cell 6: LangChain GenAI pipeline
gen_pipeline = pipeline("text2text-generation", model="google/flan-t5-small", max_new_tokens=128)
llm = HuggingFacePipeline(pipeline=gen_pipeline)

prompt = PromptTemplate.from_template(
    "Write a creative, vivid but concise product blurb (70-100 words) for a furniture item.\n"
    "Title: {title}\nBrand: {brand}\nMaterial: {material}\nColor: {color}\nCategories: {categories}\n"
    "Make it friendly and helpful for shoppers."
)

# Test on a sample
sample = df.iloc[0]
print(llm(prompt.format(
    title=sample['title'],
    brand=sample['brand'],
    material=sample.get('material',''),
    color=sample.get('color',''),
    categories=' > '.join(sample['categories'][:4]) if isinstance(sample['categories'], list) else ''
)))


In [ ]:
import os, joblib, pandas as pd
os.makedirs('backend/models', exist_ok=True)
os.makedirs('backend/data', exist_ok=True)

# Replace with your actual variables from training
joblib.dump(clf, 'backend/models/clip_linear_classifier.joblib')
joblib.dump(label_space, 'backend/models/label_space.joblib')
df[['uniq_id','cluster']].to_csv('backend/data/clusters.csv', index=False)

# Ensure the provided dataset is copied for backend analytics
!cp intern_data_ikarus.csv backend/data/intern_data_ikarus.csv

In [ ]:
!git init
!git add .
!git commit -m "Initial commit: Initial commit: backend, frontend, artifacts, data"
!git branch -M main
!git remote add origin https://<PAT>@github.com/<username>/AI_Recommendation_App.git
!git push -u origin main

In [ ]:
!git config --global user.email "asharma4_be22@thapar.edu"
!git config --global user.name "asharmaaryamani"

In [ ]:
# In Colab, with your repo cloned as the working directory
import os, shutil
os.makedirs('backend/data', exist_ok=True)
shutil.copy('intern_data_ikarus.csv', 'backend/data/intern_data_ikarus.csv')

# then commit and push
!git add backend/data/intern_data_ikarus.csv
!git commit -m "Add dataset CSV for API analytics and indexing"
!git push
